In [16]:
#!/usr/bin/env python3
"""plot_rq4.py — RQ4 shard scaling (Figure 9), one file per panel.

    python3 plot_rq4.py  results_rq4  rq4_figs            [--grid]

  In Jupyter: run this cell, then in a new cell
    main(["results_rq4", "rq4_figs"])

Reads every rq4_shard_scaling*.csv under CSV_DIR (written by RQ4_ShardScaling /
run_rq4.sh).  Several CSVs with the same shard counts (e.g. one per seed) are
averaged: point = mean, error bar = +-1 sd over the CSVs.  With a single CSV
the error bars are the p25..p75 range over the measured epochs (columns
throughput_p25/p75, epoch_ms_p25/p75).  If the CSVs hold more than one
`distribution` (uniform / skewed), every figure is written once per
distribution with the suffix _uniform / _skewed.

Writes to OUT_DIR, each as .pdf and .png:
  fig9a_throughput      committed txn/s vs k (log-log), dotted = linear from k=1
  fig9b_speedup         measured speedup vs k (log-log) and ideal linear; --model adds k * tau_1 / tau_k
  fig9c_oram_cost       epoch latency tau_k (left) and ORAM tree height (right)
  fig9_shard_scaling    the 1x3 figure (only with --grid)
  rq4_summary.csv       the numbers behind the figures
"""
import argparse, csv, glob, math, os, statistics, sys
from collections import defaultdict

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, NullLocator, FuncFormatter

In [17]:
plt.rcParams.update({"font.family": "serif", "font.size": 8, "axes.titlesize": 9, "axes.labelsize": 8.5,
                     "legend.fontsize": 7, "xtick.labelsize": 8, "ytick.labelsize": 8,
                     "pdf.fonttype": 42, "ps.fonttype": 42, "mathtext.fontset": "dejavuserif"})

BLUE, RED, GREEN, BLACK = "#1f77b4", "#d62728", "#2ca02c", "black"

In [18]:
def f(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return float("nan")

In [19]:
def load(csv_dir):
    files = sorted(glob.glob(os.path.join(csv_dir, "rq4_shard_scaling*.csv")))
    if not files:   # zip unpacked one level deeper, or a single csv passed directly
        files = sorted(glob.glob(os.path.join(csv_dir, "**", "rq4_shard_scaling*.csv"), recursive=True))
    if not files and os.path.isfile(csv_dir):
        files = [csv_dir]
    data = defaultdict(lambda: defaultdict(list))    # dist -> k -> [row per csv]
    for path in files:
        with open(path) as fh:
            for r in csv.DictReader(fh):
                data[r.get("distribution", "uniform") or "uniform"][int(f(r["shards"]))].append(r)
    print(f"  {len(files)} csv file(s):", *[os.path.basename(p) for p in files])
    return data

In [20]:
def agg(rows, key, lo_key=None, hi_key=None):
    """mean and (lower, upper) error: sd over CSVs, or p25/p75 with one CSV."""
    v = [f(r[key]) for r in rows if f(r[key]) == f(r[key])]
    if not v:
        return float("nan"), 0.0, 0.0
    m = statistics.fmean(v)
    if len(v) > 1:
        s = statistics.stdev(v)
        return m, s, s
    if lo_key and hi_key and lo_key in rows[0] and hi_key in rows[0]:
        lo, hi = f(rows[0][lo_key]), f(rows[0][hi_key])
        if lo == lo and hi == hi and lo > 0:
            return m, max(0.0, m - lo), max(0.0, hi - m)
    return m, 0.0, 0.0

In [21]:
def log2_axis(ax, ks):
    ax.set_xscale("log", base=2)
    ax.xaxis.set_major_locator(FixedLocator(ks))
    ax.xaxis.set_minor_locator(NullLocator())
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x:g}"))
    ax.set_xlim(ks[0] / 1.25, ks[-1] * 1.25)
    ax.set_xlabel("Number of shards $k$")
    ax.grid(True, alpha=0.3, linewidth=0.6)

In [22]:
def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("csv_dir", help="folder with rq4_shard_scaling*.csv (or one csv file)")
    ap.add_argument("out")
    ap.add_argument("--grid", action="store_true", help="also write the 1x3 figure fig9_shard_scaling")
    ap.add_argument("--width", type=float, default=2.6)
    ap.add_argument("--height", type=float, default=2.2)
    ap.add_argument("--label", default="SidePart", help="legend name of the system")
    ap.add_argument("--model", action="store_true", help="speedup panel: also draw the model k * tau_1 / tau_k")
    a = ap.parse_args(argv)       # argv=None -> command line; in Jupyter pass a list

    data = load(a.csv_dir)
    if not data:
        print("!! no rq4_shard_scaling*.csv in", os.path.abspath(a.csv_dir), " (cwd:", os.getcwd() + ")")
        return
    os.makedirs(a.out, exist_ok=True)
    print("  output ->", os.path.abspath(a.out))

    def save(fig, name):
        fig.tight_layout(pad=0.3)
        for ext in ("pdf", "png"):
            fig.savefig(os.path.join(a.out, f"{name}.{ext}"), dpi=300, bbox_inches="tight")
        plt.close(fig)
        print("  wrote", name + ".pdf/.png")

    summary = []
    multi = len(data) > 1
    for dist, by_k in sorted(data.items()):
        sfx = f"_{dist}" if multi else ""
        ks = sorted(by_k)
        T = [agg(by_k[k], "throughput_txn_s", "throughput_p25", "throughput_p75") for k in ks]
        L = [agg(by_k[k], "epoch_ms_median", "epoch_ms_p25", "epoch_ms_p75") for k in ks]
        H = [agg(by_k[k], "mean_tree_height") for k in ks]
        k1 = ks[0]
        t1, l1 = T[0][0], L[0][0]
        speed = [t[0] / t1 if t1 > 0 else float("nan") for t in T]
        # error of the speedup ratio from the throughput error at k (k1 taken as exact)
        speed_e = [(t[1] / t1, t[2] / t1) if t1 > 0 else (0, 0) for t in T]
        model = [(k / k1) * l1 / l[0] if l[0] > 0 else float("nan") for k, l in zip(ks, L)]
        linear = [k / k1 for k in ks]

        for k, t, l, h, s, mo in zip(ks, T, L, H, speed, model):
            r0 = by_k[k][0]
            summary.append([dist, k, len(by_k[k]), round(t[0], 2), round(l[0], 2), round(h[0], 2),
                            round(s, 3), round(mo, 3), round(s / (k / k1), 3),
                            r0.get("fanout", ""), r0.get("imbalance_queries", ""), r0.get("commit_rate", ""),
                            r0.get("abort_read_batch_full", ""), r0.get("partitions", "")])
            if r0.get("partitions") and int(f(r0["partitions"])) != k:
                print(f"  !! {dist} k={k}: partitioner produced {r0['partitions']} partitions")

        # ---- panel drawers --------------------------------------------------
        def draw_tput(ax):
            ax.plot(ks, [t1 * x for x in linear], color=BLACK, linestyle=":", linewidth=1,
                    label=f"linear (from $k$={k1})")
            ax.errorbar(ks, [t[0] for t in T], yerr=[[t[1] for t in T], [t[2] for t in T]], color=BLUE,
                        marker="o", markersize=5, linewidth=1.5, capsize=3, label=a.label)
            ax.set_yscale("log")
            from matplotlib.ticker import LogLocator, NullFormatter
            ax.yaxis.set_major_locator(LogLocator(base=10, subs=(1, 2, 5)))
            ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y:,.0f}"))
            ax.yaxis.set_minor_formatter(NullFormatter())
            ax.set_ylabel("Committed txn/s")
            log2_axis(ax, ks)
            ax.legend(frameon=False, loc="upper left")

        def draw_speedup(ax):
            ax.errorbar(ks, speed, yerr=[[e[0] for e in speed_e], [e[1] for e in speed_e]], color=BLUE,
                        marker="o", markersize=5, linewidth=1.5, capsize=3, label="measured")
            if a.model:
                ax.plot(ks, model, color=RED, linestyle="--", marker="s", markersize=5,
                        markerfacecolor="white", linewidth=1.5, label="model $k\\,\\tau_1/\\tau_k$")
            ax.plot(ks, linear, color=BLACK, linestyle=":", linewidth=1, label="linear")
            ax.set_yscale("log", base=2)
            top = max([x for x in speed + (model if a.model else []) + linear if x == x])
            ticks = [2 ** i for i in range(0, int(math.ceil(math.log2(max(top, 1)))) + 1)]
            ax.yaxis.set_major_locator(FixedLocator(ticks))
            ax.yaxis.set_minor_locator(NullLocator())
            ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y:g}"))
            ax.set_ylim(0.85, ticks[-1] * 1.08)
            ax.set_ylabel(f"Speedup over $k$={k1}")
            log2_axis(ax, ks)
            h, l = ax.get_legend_handles_labels()          # errorbar handles come last: put "measured" first
            order = [l.index("measured")] + [i for i in range(len(l)) if l[i] != "measured"]
            ax.legend([h[i] for i in order], [l[i] for i in order], frameon=False, loc="upper left")

        def draw_cost(ax):
            h1 = ax.errorbar(ks, [l[0] for l in L], yerr=[[l[1] for l in L], [l[2] for l in L]], color=BLACK,
                             marker="o", markersize=5, linewidth=1.5, capsize=3, label="epoch latency $\\tau_k$")
            ax.set_ylabel("Epoch latency (ms)")
            ax.set_ylim(0, max(l[0] + l[2] for l in L) * 1.12)
            log2_axis(ax, ks)
            ax2 = ax.twinx()
            h2, = ax2.plot(ks, [h[0] for h in H], color=GREEN, linestyle="--", marker="^", markersize=5,
                           linewidth=1.5, label="tree height")
            ax2.set_ylabel("ORAM tree height", color=GREEN)
            ax2.tick_params(axis="y", colors=GREEN)
            ax2.set_ylim(0, 5 * math.ceil((max(h[0] for h in H) + 1) / 5))
            ax.legend([h1, h2], [h1.get_label(), h2.get_label()], frameon=False, loc="lower left")

        drawers = [("fig9a_throughput", "(a) Throughput at saturation", draw_tput),
                   ("fig9b_speedup", "(b) Speedup", draw_speedup),
                   ("fig9c_oram_cost", "(c) Per-shard ORAM cost", draw_cost)]

        for name, _title, fn in drawers:           # single panels (no title: use \subcaption)
            fig, ax = plt.subplots(figsize=(a.width, a.height))
            fn(ax)
            save(fig, name + sfx)

        if a.grid:
            fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.3))
            for ax, (_n, title, fn) in zip(axes, drawers):
                fn(ax)
                ax.set_title(title)
            save(fig, "fig9_shard_scaling" + sfx)

    with open(os.path.join(a.out, "rq4_summary.csv"), "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["distribution", "shards", "n_csv", "throughput_txn_s", "epoch_ms", "tree_height",
                    "speedup", "model_speedup", "efficiency", "fanout", "imbalance_queries", "commit_rate",
                    "abort_read_batch_full", "partitions"])
        w.writerows(summary)
    print("  wrote rq4_summary.csv")
    print(f"  {'dist':8s} {'k':>3s} {'txn/s':>9s} {'tau_k ms':>9s} {'height':>6s} {'speedup':>8s} {'model':>6s}")
    for r in summary:
        print(f"  {r[0]:8s} {r[1]:3d} {r[3]:9.1f} {r[4]:9.1f} {r[5]:6.1f} {r[6]:8.2f} {r[7]:6.2f}")


if __name__ == "__main__" and "ipykernel" not in sys.modules:
    main()

In [ ]:
main(["results_rq4", "rq4_figs", "--grid"])